In [1]:
# DQN plays first, training 'O' with a stable win rate (plays as O).  
import numpy as np
import pandas as pd
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import torch.optim as optim
import math
from itertools import count
import wandb

# Initialize WandB
wandb.login(key='e21de1f4d4c13b4ba109db92ba20cc946e7da3c5')
wandb.init(project="experiments", name="connect4 dqn going first")

class ConnectX:
    def __init__(self, height=6, width=7):
        self.board_height = height
        self.board_width = width
        self.board_state = np.zeros((height, width), dtype=np.int8)
        self.players = {'p1': 1, 'p2': 2}
        self.isDone = False
        self.reward = {'win': 1, 'draw': 0.5, 'lose': -1, 'step': -0.01}

    def render(self):
        rendered_board = self.board_state.astype(str)
        rendered_board[self.board_state == 0] = ' '
        rendered_board[self.board_state == 1] = 'O'
        rendered_board[self.board_state == 2] = 'X'
        display(pd.DataFrame(rendered_board))

    def reset(self):
        self.board_state = np.zeros((self.board_height, self.board_width), dtype=np.int8)
        self.isDone = False
        return self.board_state.copy()

    def get_available_actions(self):
        return [j for j in range(self.board_width) if self.board_state[0, j] == 0]

    def check_win(self, player):
        player_id = self.players[player]
        board = self.board_state
        ##check horizontally
        for i in range(self.board_height):
            for j in range(self.board_width - 3):
                if np.all(board[i, j:j+4] == player_id):
                    return True
        ##check vertically
        for j in range(self.board_width):
            for i in range(self.board_height - 3):
                if np.all(board[i:i+4, j] == player_id):
                    return True
        ##Diagonal checks
        for i in range(self.board_height - 3):
            ##top-left to bottom-right
            for j in range(self.board_width - 3):
                if np.all([board[i+k, j+k] == player_id for k in range(4)]):
                    return True
            ##top-right to bottom-left
            for j in range(3, self.board_width):
                if np.all([board[i+k, j-k] == player_id for k in range(4)]):
                    return True
        return False

    def check_game_done(self, player):
        if self.check_win(player):
            self.isDone = True
            return self.reward['win']
        if not np.any(self.board_state == 0):
            self.isDone = True
            return self.reward['draw']
        return self.reward['step']

    def make_move(self, action, player):
        ##If move is illegal and columns are full, player losses
        if action not in self.get_available_actions():
            return self.board_state.copy(), self.reward['lose'], False
        #Finds the lowest empty spot in the chosen column.
        i = self.board_height - 1 - np.sum(self.board_state[:, action] != 0)
        self.board_state[i, action] = self.players[player]
        #Check if the move ended the game.
        reward = self.check_game_done(player)
        return self.board_state.copy(), reward, True

class ReplayMemory:
    def __init__(self, capacity=100000):
        self.memory = []
        self.capacity = capacity
        self.position = 0

    def dump(self, transition):
        if len(self.memory) < self.capacity:
            self.memory.append(None)
        self.memory[self.position] = transition
        ##Move to the next slot. If at the end, loop back to 0 (overwriting old stuff).
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

class DQN(nn.Module):
    def __init__(self, outputs, height=6, width=7):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(height * width, 128)  # 42 inputs
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, outputs)  # 7 outputs

    def forward(self, x):
        x = x.view(x.size(0), -1)  
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# Hyperparameters
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.05
EPS_DECAY = 1000
TARGET_UPDATE = 10
MEMORY_CAPACITY = 100000
NUM_EPISODES = 30000
LOG_INTERVAL = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

env = ConnectX()
memory = ReplayMemory(MEMORY_CAPACITY)
policy_net = DQN(env.board_width).to(device)
target_net = DQN(env.board_width).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()
optimizer = optim.Adam(policy_net.parameters(), lr=5e-4)

def select_action(state, available_actions, steps_done, training=True):
    ##Converts the board (state) into a tensor
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(0)
    ###decides whether to exploit or explore.
    if training:
        eps_threshold = EPS_END + (EPS_START - EPS_END) * math.exp(-steps_done / EPS_DECAY)
    else:
        eps_threshold = 0
    if random.random() > eps_threshold:
        with torch.no_grad():
            ##Get Q-values for only the available columns. Pick the column with the highest score (best move).
            q_values = policy_net(state)[0, available_actions]
            return available_actions[torch.argmax(q_values).item()]
    ##else pick a random column
    return random.choice(available_actions)

def optimize_model():
    ##Check if there's enough memory to learn
    if len(memory) < BATCH_SIZE:
        return
    transitions = memory.sample(BATCH_SIZE)
    ###If any moves were invalid (maybe due to wrong actions), skip learning with those.
    valid_transitions = [t for t in transitions if t[1] is not None]
    if len(valid_transitions) < BATCH_SIZE:
        return
    ###Separate all states, actions, rewards, next states into their own lists.
    batch = list(zip(*valid_transitions))
    state_batch = torch.stack([torch.tensor(state, dtype=torch.float32).unsqueeze(0) for state in batch[0]], dim=0).to(device)
    action_batch = torch.tensor(batch[1], dtype=torch.long, device=device)
    reward_batch = torch.tensor(batch[2], dtype=torch.float32, device=device)
    next_state_batch = [s for s in batch[3] if s is not None]
    non_final_mask = torch.tensor([s is not None for s in batch[3]], device=device)

     
    ###(predicted Q-values)
    state_action_values = policy_net(state_batch).gather(1, action_batch.unsqueeze(1))
    ###best Q-value from next states. 
    next_state_values = torch.zeros(len(valid_transitions), device=device)
    if next_state_batch:
        next_states = torch.stack([torch.tensor(s, dtype=torch.float32).unsqueeze(0) for s in next_state_batch], dim=0).to(device)
        next_state_values[non_final_mask] = target_net(next_states).max(1)[0].detach()
    expected_state_action_values = reward_batch + (GAMMA * next_state_values)
 
    ###Calculate loss (difference between prediction and reality)
    loss = F.smooth_l1_loss(state_action_values, expected_state_action_values.unsqueeze(1))
    optimizer.zero_grad()  # Clear old gradients
    loss.backward()        # Calculate new gradients (how to change weights)
    optimizer.step()       # Update the network weights 
    return loss.item() if loss is not None else 0

def random_agent(actions):
    return random.choice(actions)

def evaluate_policy(episodes=100):
    wins, losses, draws, moves_taken = 0, 0, 0, []
    for _ in range(episodes):
        state = env.reset()
        move_count = 0
        while not env.isDone:
            available_actions = env.get_available_actions()
            action = select_action(state, available_actions, steps_done=0, training=False)
            state, reward, valid = env.make_move(action, 'p1')
            move_count += 1
            if not valid:
                losses += 1
                break
            if env.isDone:
                if reward == 1:
                    wins += 1
                    moves_taken.append(move_count)
                elif reward == 0.5:
                    draws += 1
                break
            available_actions = env.get_available_actions()
            action = random_agent(available_actions)
            state, reward, _ = env.make_move(action, 'p2')
            move_count += 1
            if env.isDone:
                if reward == 1:
                    losses += 1
                elif reward == 0.5:
                    draws += 1
                break
    return wins / episodes, losses / episodes, draws / episodes, np.mean(moves_taken) if moves_taken else 0

# Training loop with corrected in-training rate calculations from 'O' perspective
steps_done = 0
training_history = []
best_win_rate = 0
interval_wins = 0    # 'O' wins
interval_losses = 0  # 'O' losses (when 'X' wins or invalid move by 'O')
interval_draws = 0   # Draws
interval_total_moves = 0  # Total moves in games within the interval

for episode in range(NUM_EPISODES):
    state = env.reset()
    move_count = 0
    outcome = None

    for t in count():
        # Agent plays as 'O' (p1)
        available_actions = env.get_available_actions()
        action = select_action(state, available_actions, steps_done, training=True)
        steps_done += 1
        next_state, reward, valid = env.make_move(action, 'p1')
        move_count += 1
        if not valid:
            outcome = 'loss'
            memory.dump((state, action, reward, None))
            # if episode % 1 == 0:
            #     print(f"Episode {episode} ended: Invalid move by 'X'!")
            #     env.render()
            break
        # if episode % 1 == 0:
        #     print(f"Episode {episode}, Move {t}, Player 'O' Action: {action}")
        #     env.render()
        if env.isDone:
            if reward == 1:  # 'O' wins
                outcome = 'win'
            elif reward == 0.5:  # Draw
                outcome = 'draw'
            # if episode % 1 == 0:
            #     print(f"Episode {episode} ended: {'O' if reward == 1 else 'Draw' if reward == 0.5 else 'Invalid'}!")
            #     env.render()
            memory.dump((state, action, reward, next_state if not env.isDone else None))
            break

        # Opponent plays as 'X' (p2)
        available_actions = env.get_available_actions()
        action_p2 = random_agent(available_actions)
        next_next_state, reward_p2, _ = env.make_move(action_p2, 'p2')
        move_count += 1
        # if episode % 1 == 0:
        #     print(f"Episode {episode}, Move {t+1}, Player 'X' Action: {action_p2}")
        #     env.render()
        if env.isDone:
            if reward_p2 == 1:  # 'X' wins
                outcome = 'loss'  # Loss for 'O'
                reward = -1  # Negative reward for 'O' when 'X' wins
            elif reward_p2 == 0.5:  # Draw
                outcome = 'draw'
                reward = -0.5  # Negative reward for 'O' when draw
            else:
                reward = env.reward['step']  # Default step penalty if not terminal
            # if episode % 1 == 0:
            #     print(f"Episode {episode} ended: {'X' if reward_p2 == 1 else 'Draw' if reward_p2 == 0.5 else 'Invalid'} wins!")
            #     env.render()
            memory.dump((state, action, reward, next_state if not env.isDone else None))
            break
        # For non-terminal states after 'X''s move, use the step penalty
        memory.dump((state, action, reward, next_state))
        state = next_next_state
        optimize_model()

    # Update interval counters based on the episode outcome
    if outcome == 'win':
        interval_wins += 1
    elif outcome == 'loss':
        interval_losses += 1
    elif outcome == 'draw':
        interval_draws += 1
    interval_total_moves += move_count

    if episode % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    # Log performance metrics every LOG_INTERVAL episodes
    if (episode + 1) % LOG_INTERVAL == 0 or episode == NUM_EPISODES - 1:
        total_games_in_interval = episode % LOG_INTERVAL + 1 if episode == NUM_EPISODES - 1 and (episode + 1) % LOG_INTERVAL != 0 else LOG_INTERVAL
        if total_games_in_interval > 0:
            interval_win_rate = interval_wins / total_games_in_interval
            interval_loss_rate = interval_losses / total_games_in_interval
            interval_draw_rate = interval_draws / total_games_in_interval
            interval_win_draw_rate = interval_win_rate + interval_draw_rate
            interval_avg_moves = interval_total_moves / total_games_in_interval

            print(f"\n--- Episode {episode + 1}/{NUM_EPISODES} ---")
            print(f"Training Summary (last {total_games_in_interval} episodes, 'O' perspective):")
            print(f" 'O' Wins: {interval_wins}, Losses: {interval_losses}, Draws: {interval_draws}")
            print(f" Win Rate: {interval_win_rate:.3f}, Loss Rate: {interval_loss_rate:.3f}, Draw Rate: {interval_draw_rate:.3f}, Win+Draw Rate: {interval_win_draw_rate:.3f}")
            print(f" Avg Moves per game: {interval_avg_moves:.1f}")

            # Log to WandB
            wandb.log({
                "Episode": episode + 1,
                "Training/Interval Wins": interval_wins,
                "Training/Interval Losses": interval_losses,
                "Training/Interval Draws": interval_draws,
                "Win Rate": interval_win_rate,
                "Loss Rate": interval_loss_rate,
                "Draw Rate": interval_draw_rate,
                "Win+Draw Rate": interval_win_draw_rate,
                "Training/Interval Avg Moves": interval_avg_moves})

        # Reset interval counters
        interval_wins = 0
        interval_losses = 0
        interval_draws = 0
        interval_total_moves = 0

wandb.finish()
print("Training complete")
final_model_path = "../saved_models/connect4_dqn_first_final.pth"
torch.save(policy_net.state_dict(), final_model_path)
print(f"Trained model saved to '{final_model_path}'")

def demo():
    env.reset()
    env.render()
    while not env.isDone:
        state = env.board_state.copy()
        available_actions = env.get_available_actions()
        action = select_action(state, available_actions, steps_done=0, training=False)
        state, reward, valid = env.make_move(action, 'p1')
        env.render()
        if not valid:
            print("Invalid move by 'O'!")
            break
        if reward == 1:
            print("O Wins!")
            break
        if reward == 0.5:
            print("Draw!")
            break
        available_actions = env.get_available_actions()
        action = random_agent(available_actions)
        state, reward, _ = env.make_move(action, 'p2')
        env.render()
        if reward == 1:
            print("X Wins!")
            break
        elif reward == 0.5:
            print("Draw!")
            break

demo()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/saideepa0501/.netrc
wandb: Currently logged in as: kradeero (kradeero-ohio-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using device: cpu

--- Episode 100/30000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 64, Losses: 36, Draws: 0
 Win Rate: 0.640, Loss Rate: 0.360, Draw Rate: 0.000, Win+Draw Rate: 0.640
 Avg Moves per game: 21.6

--- Episode 200/30000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 69, Losses: 31, Draws: 0
 Win Rate: 0.690, Loss Rate: 0.310, Draw Rate: 0.000, Win+Draw Rate: 0.690
 Avg Moves per game: 19.8

--- Episode 300/30000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 51, Losses: 48, Draws: 1
 Win Rate: 0.510, Loss Rate: 0.480, Draw Rate: 0.010, Win+Draw Rate: 0.520
 Avg Moves per game: 20.0

--- Episode 400/30000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 52, Losses: 48, Draws: 0
 Win Rate: 0.520, Loss Rate: 0.480, Draw Rate: 0.000, Win+Draw Rate: 0.520
 Avg Moves per game: 19.9

--- Episode 500/30000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 58, Losses: 4

Draw Rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Episode,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
Loss Rate,▄▇▅▄▆▃▇▅▄▃▄▆▅▄▅▄▃▆▅▄▄▂▃▁▆▅▆▄█▆▃▅▆▅▅▃▄▄▄▁
Training/Interval Avg Moves,█▃▃▁▂▂▁▂▂▁▂▂▂▂▁▂▃▂▁▂▁▂▂▁▂▁▂▁▁▂▂▂▂▂▂▃▁▁▁▂
Training/Interval Draws,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Training/Interval Losses,▇█▅▂▂▂▂▂▃▃▂▃▂▃▃▃▃▃▃▂▃▂▃▃▂▃▁▃▂▃▃▃▃▂▂▃▃▂▂▂
Training/Interval Wins,▆▇▄▆▅▅▇▅▃▄▆▆▇▇▄▆▃▃▃▅▆▃▃▇▁▃▆▃▅▅▅▃▆▃▇█▆▅▅▅
Win Rate,▂▁▂▇▂▃▇▄▂▃▃▅▂▇▇▃▃▆▅▂▅▆▂█▂▃▂▅▂▅▅▂▃▄▅▅▃▆▇▄
Win+Draw Rate,▁▆▆▇▆▇▇▆▇▇▇▇▇▆▆▇▆▆▇▆▇▅▆▆▇▇▆▇█▆▆▆▆█▆▆▆▇▅▆
Draw Rate,0
Episode,30000


Training complete
Trained model saved to '../saved_models/connect4_dqn_first_final.pth'


,0,1,2,3,4,5,6
0,,,,,,,
1,,,,,,,
2,,,,,,,
3,,,,,,,
4,,,,,,,
5,,,,,,,


...

,0,1,2,3,4,5,6
0,,,,,,,
1,,,,,,,
2,,,,,,,
3,,,,,,,
4,,,,,O,,
5,,X,,,O,,X


,0,1,2,3,4,5,6
0,,,,,,,
1,,,,,,,
2,,,,,,,
3,,,,,O,,
4,,,,,O,,
5,,X,,,O,,X


,0,1,2,3,4,5,6
0,,,,,,,
1,,,,,,,
2,,,,,,,
3,,,,,O,,
4,,,,,O,,
5,,X,X,,O,,X


,0,1,2,3,4,5,6
0,,,,,,,
1,,,,,,,
2,,,,,O,,
3,,,,,O,,
4,,,,,O,,
5,,X,X,,O,,X


O Wins!
